# Phase 6a: Scale-Up: Non-Redundant Ingestion at Thousands of Chains

The Phase 5b result (topology +0.054 PR-AUC, 4/5 folds) is consistent but
**underpowered**: with 21 sequence clusters, 5 folds cannot reach p < 0.05 even
for a real effect, the minimum attainable p-value is 0.0625. Sample size, not
method, is the binding constraint. This notebook removes it.

---

### On the data source

The original plan named BioLiP. **BioLiP's site returns HTTP 403 to programmatic
requests** (verified against `zhanggroup.org/BioLiP/` and its download paths), so
depending on it would make the pipeline fragile. Instead this builds an
equivalent set directly from RCSB, which is fully programmatic and unauthenticated:

- **Selection** via the RCSB Search API, X-ray structures, resolution ≤ 2.5 Å,
  ≥ 1 protein entity, ≥ 1 non-polymer entity, 50–1500 residues. That query
  returns **126,921 entries**, and the ligand constraint is doing real work
  (163,564 → 138,114 before the length filter, so ~25k apo structures are removed).
- **Labelling** uses the *same BioLiP convention* already validated in Phase 1:
  a residue is positive if any heavy atom lies within 4.5 Å of a biologically
  relevant ligand, with cryoprotectants and buffer components screened out.

So the scientific definition is unchanged; only the delivery mechanism differs.

### Non-redundancy is the point

Entries are sampled with a **seeded shuffle** before truncation. Taking the first
N from the API would return `101M, 102L, 103L, ...`, alphabetical order is
deposition order, so an unshuffled prefix is entirely 1990s structures.

After clustering at 30 % identity, **one representative chain per cluster** is
kept. This is what buys statistical power: 2,000 independent clusters instead of
21, and it simultaneously cuts the Phase 6b topology compute by removing
redundant homologues that would otherwise contribute nothing but runtime.

## Cell 1: Config

In [ ]:
!pip install -q torch-geometric==2.6.1 biopython==1.84

In [ ]:
import gzip, json, os, random, shutil, subprocess, time, urllib.request, warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")

@dataclass
class Cfg:
    n_target:         int   = 6000      # entries to download (of ~127k available)
    max_resolution:   float = 2.5
    contact_cutoff:   float = 8.0
    max_neighbors:    int   = 32
    min_residues:     int   = 50
    max_residues:     int   = 1000
    ligand_cutoff:    float = 4.5
    min_ligand_atoms: int   = 6
    include_metals:   bool  = True
    min_positives:    int   = 3
    identity_thresh:  float = 0.30
    workers:          int   = 16
    seed:             int   = 1337

CFG = Cfg()
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)
OUT = Path("/kaggle/working/processed"); OUT.mkdir(parents=True, exist_ok=True)
random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)
print(f"target {CFG.n_target} entries | {os.cpu_count()} CPUs")

## Cell 2: Select entries via the RCSB Search API

`return_all_hits` pulls every matching ID in one request (~1.5 s for 127k), then
a seeded shuffle makes the truncation an unbiased sample rather than a slice of
1990s depositions.

In [ ]:
QUERY = {
  "query": {"type": "group", "logical_operator": "and", "nodes": [
    {"type": "terminal", "service": "text", "parameters": {
      "attribute": "rcsb_entry_info.resolution_combined",
      "operator": "less_or_equal", "value": CFG.max_resolution}},
    {"type": "terminal", "service": "text", "parameters": {
      "attribute": "rcsb_entry_info.nonpolymer_entity_count",
      "operator": "greater_or_equal", "value": 1}},
    {"type": "terminal", "service": "text", "parameters": {
      "attribute": "rcsb_entry_info.polymer_entity_count_protein",
      "operator": "greater_or_equal", "value": 1}},
    {"type": "terminal", "service": "text", "parameters": {
      "attribute": "rcsb_entry_info.deposited_polymer_monomer_count",
      "operator": "range", "value": {"from": CFG.min_residues, "to": 1500}}},
  ]},
  "return_type": "entry",
  "request_options": {"return_all_hits": True, "results_content_type": ["experimental"]},
}

t0 = time.time()
req = urllib.request.Request("https://search.rcsb.org/rcsbsearch/v2/query",
                             data=json.dumps(QUERY).encode(),
                             headers={"Content-Type": "application/json"})
hits = json.load(urllib.request.urlopen(req, timeout=300))
all_ids = [h["identifier"] for h in hits["result_set"]]
print(f"{len(all_ids):,} entries match the quality filter ({time.time() - t0:.1f}s)")

rng = random.Random(CFG.seed)
rng.shuffle(all_ids)                       # unbiased sample, not a deposition-era slice
target_ids = sorted(all_ids[:CFG.n_target])
print(f"sampled {len(target_ids):,} | example: {target_ids[:6]}")

## Cell 3: Download

Gzipped transfers (~10× smaller). `.pdb.gz` first, `.cif.gz` fallback for the
entries too large for the legacy format. Cached, so re-running costs nothing.

In [ ]:
def fetch(pid: str):
    for ext in ("pdb", "cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500:
            return pid, p
    for ext in ("pdb", "cif"):
        url = f"https://files.rcsb.org/download/{pid}.{ext}.gz"
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                data = r.read()
            if len(data) > 500:
                p = RAW / f"{pid}.{ext}.gz"; p.write_bytes(data)
                return pid, p
        except Exception:
            continue
    return pid, None


t0 = time.time()
with ThreadPoolExecutor(max_workers=CFG.workers) as ex:
    got = list(ex.map(fetch, target_ids))
paths = {p: f for p, f in got if f is not None}
mb = sum(f.stat().st_size for f in paths.values()) / 1e6
print(f"downloaded {len(paths):,}/{len(target_ids):,} in {(time.time()-t0)/60:.1f} min "
      f"({mb:.0f} MB)")

## Cell 4: Parse and label

Identical predicates and ligand screening to Phase 1, the code that produced a
9.17 % positive rate and 96 % label contiguity on the 32-structure set. Parsing
is parallel and failure-tolerant: at this scale some entries will always be
malformed, and one exception must not kill a 20-minute job.

In [ ]:
from Bio.PDB import MMCIFParser, PDBParser

AA3_TO_1 = {"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E",
            "GLY":"G","HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F",
            "PRO":"P","SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
NON_LIGAND = {"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD",
              "TRS","ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI",
              "CO3","IOD","BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS = {"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}


def _heavy(res):
    xyz = [a.coord for a in res.get_unpacked_list() if a.element not in ("H", "D")]
    return np.asarray(xyz, dtype=np.float32).reshape(-1, 3)


def parse_one(pid, path):
    try:
        opener = gzip.open(path, "rt")
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with opener as fh:
            model = next(parser.get_structure(pid, fh).get_models())
    except Exception:
        return []

    lig = []
    for ch in model:
        for res in ch:
            if res.id[0] == " ":
                continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND:
                continue
            xyz = _heavy(res)
            is_metal = nm in CATALYTIC_METALS
            if is_metal and not CFG.include_metals:
                continue
            if not is_metal and len(xyz) < CFG.min_ligand_atoms:
                continue
            lig.append(xyz)
    if not lig:
        return []
    tree = cKDTree(np.concatenate(lig, axis=0))

    out = []
    for ch in model:
        residues = [r for r in ch
                    if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
        if not (CFG.min_residues <= len(residues) <= CFG.max_residues):
            continue
        ca, seq, dmin = [], [], []
        for res in residues:
            ca.append(res["CA"].coord)
            seq.append(AA3_TO_1.get(res.get_resname().strip().upper(), "X"))
            atoms = _heavy(res)
            dmin.append(float(tree.query(atoms, k=1)[0].min()) if len(atoms) else np.inf)
        dmin = np.asarray(dmin, dtype=np.float32)
        y = (dmin <= CFG.ligand_cutoff).astype(np.float32)
        if y.sum() < CFG.min_positives:
            continue
        out.append({"pdb_id": pid, "chain_id": ch.id, "seq": "".join(seq),
                    "ca": np.asarray(ca, dtype=np.float32), "y": y, "lig_dist": dmin})
    return out


from joblib import Parallel, delayed

t0 = time.time()
batches = Parallel(n_jobs=max(1, os.cpu_count() or 2), verbose=1)(
    delayed(parse_one)(pid, p) for pid, p in sorted(paths.items()))
records = [r for b in batches for r in b]

n_res = sum(len(r["y"]) for r in records)
n_pos = sum(int(r["y"].sum()) for r in records)
print(f"\nparsed in {(time.time()-t0)/60:.1f} min")
print(f"chains {len(records):,} | residues {n_res:,} | positives {n_pos:,} "
      f"({100*n_pos/max(n_res,1):.2f}%)")
print(f"entries with >=1 usable chain: {len({r['pdb_id'] for r in records}):,}"
      f" / {len(paths):,} downloaded")

## Cell 5: MMseqs2 clustering and representative selection

MMseqs2 was validated in Phase 4 against exact Biopython all-versus-all at
**98.8 % pairwise agreement**, with every disagreement sitting at 0.30–0.31
identity, right on the threshold. All-versus-all is O(n²) and infeasible here,
so that earlier validation is what licenses using the fast path now.

Representative = **longest chain in the cluster**, ties broken by ID. Chosen
deliberately over "most positives", which would bias the set toward large pockets
and inflate every downstream metric.

In [ ]:
seqs = {f"{r['pdb_id']}_{r['chain_id']}": r["seq"] for r in records}
by_id = {f"{r['pdb_id']}_{r['chain_id']}": r for r in records}

mm = None
try:
    subprocess.run("curl -sL https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz "
                   "-o /tmp/mm.tar.gz && tar xzf /tmp/mm.tar.gz -C /tmp",
                   shell=True, check=True, timeout=600)
    cand = Path("/tmp/mmseqs/bin/mmseqs")
    if cand.exists():
        os.chmod(cand, 0o755); mm = str(cand)
        print("mmseqs2 ready")
except Exception as e:
    print(f"mmseqs2 unavailable: {type(e).__name__}")

clusters = None
if mm:
    work = Path("/tmp/mmwork")
    shutil.rmtree(work, ignore_errors=True); work.mkdir()
    (work / "in.fasta").write_text("".join(f">{k}\n{v}\n" for k, v in seqs.items()))
    t0 = time.time()
    subprocess.run([mm, "easy-cluster", str(work / "in.fasta"), str(work / "res"),
                    str(work / "tmp"), "--min-seq-id", str(CFG.identity_thresh),
                    "-c", "0.8", "--cov-mode", "1", "-v", "1"],
                   check=True, capture_output=True, timeout=3600)
    rep = defaultdict(list)
    for line in (work / "res_cluster.tsv").read_text().splitlines():
        a, b = line.split("\t"); rep[a].append(b)
    clusters = {f"C{i:05d}": v for i, v in
                enumerate(sorted(rep.values(), key=lambda v: (-len(v), v[0])))}
    print(f"clustered {len(seqs):,} chains -> {len(clusters):,} clusters "
          f"in {time.time()-t0:.1f}s")

if clusters is None:
    raise RuntimeError("MMseqs2 required at this scale; all-vs-all is infeasible")

# --- one representative per cluster: longest sequence, ties by id -------------
reps = {k: max(v, key=lambda c: (len(seqs[c]), c)) for k, v in clusters.items()}
sizes = np.array([len(v) for v in clusters.values()])
print(f"cluster sizes: median {int(np.median(sizes))}, max {sizes.max()}, "
      f"singletons {(sizes == 1).sum():,}")
print(f"representatives: {len(reps):,} chains "
      f"(redundancy removed: {len(seqs) - len(reps):,})")

## Cell 6: Build graphs (identical schema to Phase 1)

In [ ]:
from torch_geometric.data import Data

AA_ALPHABET = "ACDEFGHIKLMNPQRSTVWYX"
AA_TO_IDX = {a: i for i, a in enumerate(AA_ALPHABET)}


def radius_graph_capped(pos, r, max_neighbors=32):
    n = pos.size(0)
    pos = pos - pos.mean(dim=0, keepdim=True)      # centring: see Phase 1
    dist = torch.cdist(pos, pos).clone()
    dist.fill_diagonal_(float("inf"))
    mask = dist <= r
    if max_neighbors is not None and max_neighbors < n:
        _, knn = torch.topk(dist, max_neighbors, dim=1, largest=False)
        keep = torch.zeros_like(mask); keep.scatter_(1, knn, True); mask &= keep
    mask = mask | mask.t()
    tgt, src = mask.nonzero(as_tuple=True)
    return torch.stack([src, tgt], dim=0).long()


def build(rec, cluster_id):
    pos = torch.from_numpy(rec["ca"])
    pos = pos - pos.mean(dim=0, keepdim=True)
    n = pos.size(0)
    onehot = torch.zeros(n, len(AA_ALPHABET))
    onehot[torch.arange(n), torch.tensor([AA_TO_IDX.get(c, 20) for c in rec["seq"]])] = 1.0
    # NOTE: phi/psi are omitted at scale -- Phase 5b showed the whole 27-dim
    # sequence block reaches only PR-AUC 0.176, and recomputing dihedrals here
    # would require a second full parse. Zero-padded to keep the schema identical.
    x = torch.cat([onehot, torch.zeros(n, 6)], dim=1)
    ei = radius_graph_capped(pos, CFG.contact_cutoff, CFG.max_neighbors)
    d = (pos[ei[0]] - pos[ei[1]]).norm(dim=1, keepdim=True)
    sep = (ei[0] - ei[1]).abs().float().unsqueeze(1)
    data = Data(x=x, pos=pos, edge_index=ei,
                edge_attr=torch.cat([d, torch.log1p(sep)], dim=1),
                y=torch.from_numpy(rec["y"]),
                lig_dist=torch.from_numpy(rec["lig_dist"]))
    data.pdb_id, data.chain_id = rec["pdb_id"], rec["chain_id"]
    data.seq, data.cluster, data.num_nodes = rec["seq"], cluster_id, n
    return data


t0 = time.time()
dataset = [build(by_id[c], k) for k, c in reps.items() if c in by_id]
print(f"built {len(dataset):,} graphs in {time.time()-t0:.1f}s")

nn_ = np.array([d.num_nodes for d in dataset])
ne = np.array([d.edge_index.size(1) for d in dataset])
npos = np.array([int(d.y.sum()) for d in dataset])
print(f"residues {nn_.sum():,} | edges {ne.sum():,} (mean degree {ne.sum()/nn_.sum():.1f})")
print(f"positives {npos.sum():,} = {100*npos.sum()/nn_.sum():.2f}%")
POS_WEIGHT = (nn_.sum() - npos.sum()) / max(npos.sum(), 1)
print(f"pos_weight {POS_WEIGHT:.2f}")

## Cell 7: Splits, QC, save

In [ ]:
# --- cluster-disjoint split, greedily balanced on POSITIVES ------------------
TARGET = {"train": 0.70, "val": 0.15, "test": 0.15}
got = {k: 0 for k in TARGET}
order = sorted(dataset, key=lambda d: -int(d.y.sum()))
for d in order:
    s = min(TARGET, key=lambda k: got[k] / TARGET[k])
    d.split = s; got[s] += int(d.y.sum())

print(f"{'split':<8}{'chains':>9}{'residues':>12}{'positives':>11}{'rate':>9}")
for s in TARGET:
    ds = [d for d in dataset if d.split == s]
    r = sum(int(d.num_nodes) for d in ds); p = sum(int(d.y.sum()) for d in ds)
    print(f"{s:<8}{len(ds):>9,}{r:>12,}{p:>11,}{100*p/max(r,1):>8.2f}%")

# every chain is its own cluster representative -> split is cluster-disjoint
assert len({d.cluster for d in dataset}) == len(dataset), "duplicate cluster representatives"

for d in dataset:
    assert torch.isfinite(d.x).all() and torch.isfinite(d.pos).all()
    assert d.edge_index.max() < d.num_nodes and d.edge_index.min() >= 0
print("integrity OK")

g = torch.Generator().manual_seed(CFG.seed)
Q, _ = torch.linalg.qr(torch.randn(3, 3, generator=g)); Q = Q * torch.sign(torch.det(Q))
probe = dataset[0]
a = radius_graph_capped(probe.pos, CFG.contact_cutoff, CFG.max_neighbors)
b = radius_graph_capped(probe.pos @ Q.T + torch.randn(3) * 50, CFG.contact_cutoff, CFG.max_neighbors)
print(f"SE(3) invariance: {'OK' if torch.equal(a, b) else 'FAILED'}")

p = OUT / "phase6_graphs.pt"
torch.save({"dataset": dataset, "clusters": clusters, "reps": reps,
            "node_feat_dim": int(dataset[0].x.size(1)),
            "edge_feat_dim": int(dataset[0].edge_attr.size(1)),
            "pos_weight": float(POS_WEIGHT), "aa_alphabet": AA_ALPHABET,
            "config": {k: str(v) for k, v in CFG.__dict__.items()}}, p)
print(f"saved -> {p} ({p.stat().st_size/1e6:.0f} MB)")

---

## What this changes

Phase 5b had **21 clusters**, which caps the attainable p-value at 0.0625
regardless of effect size. This produces on the order of a **thousand-plus**
independent clusters, which is what makes the topology ablation capable of
reaching significance rather than merely suggesting an effect.

### Deliberate simplification, stated plainly

Backbone dihedrals are **not** recomputed here, the node feature block is
one-hot plus six zero columns, preserving the 27-dim schema. Justification:
Phase 5b measured the entire sequence block at PR-AUC 0.176 against `full` at
0.343, so dihedrals are a small component of an already-weak arm, and recovering
them would need a second pass over 6,000 structures. If the `seq-only` arm
matters to the final writeup, recompute them, but do it knowing the cost.

### Next

**Phase 6b**, topology over this set (~6.4 ms/residue, so budget roughly an
hour), then re-run the Phase 5b nested-CV ablation unchanged. That comparison,
at this scale, is the result the project stands on.